# Fig 1b: human fLoc selectivity by stream

This notebook mirrors `F06_C.ipynb`, but uses **subject fLoc t-maps in fsaverage space** (`lh/rh.floc* tval_subjXX.mgz`) and fsaverage stream labels (`lh/rh.streams.mgz`).

In [ ]:
import numpy as np
import pandas as pd
import nibabel as nib

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from itertools import combinations
from pathlib import Path
from scipy.stats import ttest_rel

# Use Helvetica when exporting vector figures.
mpl.rcParams['pdf.use14corefonts'] = True
mpl.rcParams['ps.useafm'] = True
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Helvetica']



In [ ]:
# Parameters
subjects = [f"{i:02d}" for i in range(1, 9)]
hemis = ["lh", "rh"]

# Match the model notebook contrasts by default.
contrast_suffix = {
    "Places": "places",
    "Bodies": "bodies",
    "Faces": "faces",
}

# Optional extra fLoc contrasts if you want them later:
# contrast_suffix.update({"Characters": "characters", "Objects": "objects"})

stream_id_to_name = {
    5: "Ventral",
    6: "Lateral",
    7: "Dorsal",  # Parietal stream in NSD streams map
}

# t-threshold used for fraction-above-threshold analysis
t_threshold = 3.0
# Shared plotting order and colors
plot_order = list(contrast_suffix.keys())
stream_order = ["Dorsal", "Lateral", "Ventral"]
ROI_COLORS = ['#377E2C', '#1A1AAC', '#8C1A4C']
stream_palette = {
    'Ventral': ROI_COLORS[2],
    'Lateral': ROI_COLORS[1],
    'Dorsal': ROI_COLORS[0],
}


In [ ]:
NSDDATA_PATH = Path("/oak/stanford/groups/kalanit/biac2/kgs/projects/Dawn/NSD/data/")
label_dir = NSDDATA_PATH / "nsddata" / "freesurfer" / "fsaverage" / "label"

if not label_dir.exists():
    raise FileNotFoundError(f"Could not locate fsaverage label directory: {label_dir}")

print(f"Using label directory: {label_dir}")


In [ ]:
# Load fsaverage stream labels and build masks
stream_labels = {}
stream_masks = {}

for hemi in hemis:
    arr = np.asarray(nib.load(label_dir / f"{hemi}.streams.mgz").dataobj).squeeze()
    stream_labels[hemi] = arr
    stream_masks[hemi] = {sid: (arr == sid) for sid in stream_id_to_name}
    print(hemi, "unique stream ids:", np.unique(arr).astype(int))


In [ ]:
# Build per-subject summary of mean t-values and fraction above threshold by stream and contrast
rows = []
missing = []

for subj in subjects:
    for hemi in hemis:
        for contrast, suffix in contrast_suffix.items():
            fpath = label_dir / f"{hemi}.floc{suffix}tval_subj{subj}.mgz"
            if not fpath.exists():
                missing.append(str(fpath))
                continue

            tvals = np.asarray(nib.load(fpath).dataobj).squeeze().astype(float)

            for sid, stream_name in stream_id_to_name.items():
                mask = stream_masks[hemi][sid]
                vals = tvals[mask]
                rows.append({
                    "subject": subj,
                    "hemi": hemi,
                    "stream": stream_name,
                    "contrast": contrast,
                    "mean_t": float(np.mean(vals)) if vals.size else np.nan,
                    "n_vertices": int(vals.size),
                    "n_t_above": int(np.sum(vals > t_threshold)) if vals.size else 0,
                    "frac_t_above": float(np.mean(vals > t_threshold)) if vals.size else np.nan,
                })

if missing:
    print(f"Missing {len(missing)} files. First few:")
    for m in missing[:5]:
        print("  ", m)

df = pd.DataFrame(rows)
print(df.shape)
df.head()


In [ ]:
# Average across hemispheres per subject
mean_df = (
    df.groupby(["subject", "stream", "contrast"], as_index=False)["mean_t"]
    .mean()
)

frac_df = (
    df.groupby(["subject", "stream", "contrast"], as_index=False)[["n_vertices", "n_t_above", "frac_t_above"]]
    .mean()
)

mean_df.head()


In [ ]:
# Pairwise stream comparisons (paired t-test across subjects) on mean t-values
results = []

for contrast in contrast_suffix:
    pivot = mean_df[mean_df["contrast"] == contrast].pivot(
        index="subject", columns="stream", values="mean_t"
    )
    for a, b in combinations(stream_order, 2):
        paired = pivot[[a, b]].dropna()
        if len(paired) == 0:
            continue
        stat, pval = ttest_rel(paired[a], paired[b])
        results.append({
            "contrast": contrast,
            "stream_a": a,
            "stream_b": b,
            "n_subjects": len(paired),
            "t_stat": stat,
            "p_value": pval,
        })

results_df = pd.DataFrame(results)
results_df["p_value_bonf"] = (results_df["p_value"] * 9).clip(upper=1.0)
results_df = results_df.sort_values(["contrast", "p_value_bonf"])
results_df


In [ ]:
# Pairwise stream comparisons on fraction t > threshold (paired t-test across subjects)
frac_results = []

for contrast in contrast_suffix:
    pivot = frac_df[frac_df["contrast"] == contrast].pivot(
        index="subject", columns="stream", values="frac_t_above"
    )
    for a, b in combinations(stream_order, 2):
        paired = pivot[[a, b]].dropna()
        if len(paired) == 0:
            continue
        stat, pval = ttest_rel(paired[a], paired[b])
        frac_results.append({
            "contrast": contrast,
            "stream_a": a,
            "stream_b": b,
            "n_subjects": len(paired),
            "t_stat": stat,
            "p_value": pval,
        })

frac_results_df = pd.DataFrame(frac_results)
frac_results_df["p_value_bonf"] = (frac_results_df["p_value"] * 9).clip(upper=1.0)
frac_results_df = frac_results_df.sort_values(["contrast", "p_value_bonf"])
frac_results_df


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

# Final paper panel: bars show mean across subject-level values; error bars are +/- 1 SD.
# frac_df has one row per subject/stream/contrast, averaged across hemispheres.
sns.barplot(
    data=frac_df,
    x="contrast",
    y="frac_t_above",
    hue="stream",
    order=plot_order,
    hue_order=stream_order,
    palette=stream_palette,
    ci="sd",        # older seaborn API: mean +/- 1 SD
    capsize=0.08,
    errwidth=1.5,   # older seaborn API
    edgecolor="white",
    linewidth=4,
    ax=ax,
)

# Overlay both hemispheres for visibility; these dots are not the error-bar sample.
markers = {"lh": "o", "rh": "^"}
for hemi, marker in markers.items():
    sns.stripplot(
        data=df[df["hemi"] == hemi],
        x="contrast",
        y="frac_t_above",
        hue="stream",
        order=plot_order,
        hue_order=stream_order,
        dodge=True,
        jitter=0.15,
        marker=marker,
        size=4.5,
        edgecolor="white",
        linewidth=0.5,
        palette=stream_palette,
        ax=ax,
    )

handles, labels = ax.get_legend_handles_labels()
stream_handles = {}
for h, lab in zip(handles, labels):
    if lab in stream_order and lab not in stream_handles:
        stream_handles[lab] = h
ax.legend(stream_handles.values(), stream_handles.keys(), frameon=False, title="Stream")

ax.spines["right"].set_visible(False)
ax.spines["top"].set_visible(False)
ax.set_xlabel("Contrast")
ax.set_ylabel(f"Fraction t > {t_threshold:g}")
plt.tight_layout()
fig = ax.figure
fig.savefig("F01_B_floc_tvalues_by_subject_stream.pdf", bbox_inches="tight")   # or .svg/.eps

ax.grid(False)
ax.tick_params(colors="black", labelcolor="black")
ax.spines["left"].set_color("black")
ax.spines["bottom"].set_color("black")
plt.show()
